In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import anndata as ad
from tqdm import tqdm
from scipy.signal import savgol_filter, find_peaks
from pybaselines import whittaker


# === Core Functions ===


def create_global_spectrum_avg(X, mz_axis):
    if hasattr(X, "toarray"):  # for sparse matrix
        X = X.toarray()
    global_spectrum = np.sum(X, axis=0)
    return mz_axis, global_spectrum

"""def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):
        X = X.toarray()
    global_spectrum = np.mean(X, axis=0)  # use average instead of sum
    return mz_axis, global_spectrum"""

"""def create_global_spectrum_avg(X, mz_axis):
    if hasattr(X, "toarray"):  # Handle sparse matrix
        X = X.toarray()
    global_spectrum = np.mean(X, axis=0)  # Mean intensity per m/z across all spots
    return mz_axis, global_spectrum"""


'''def create_global_spectrum_avg(X, mz_axis):
    if hasattr(X, "toarray"):  # Handle sparse matrix
        X = X.toarray()
    
    tic = X.sum(axis=1, keepdims=True)  # Total Ion Current per spot
    tic[tic == 0] = 1  # Avoid division by zero
    X_normalized = X / tic  # Normalize by TIC
    
    global_spectrum = np.mean(X_normalized, axis=0)  # Mean intensity per m/z
    return mz_axis, global_spectrum'''


"""def detect_hdi_style_peaks(mz_axis, global_spectrum, 
                           window_length=9, polyorder=2, 
                           min_prominence=0.01, top_n=None):
    # Step 1: Savitzky-Golay smoothing
    smoothed = savgol_filter(global_spectrum, window_length=window_length, polyorder=polyorder)

    # Step 2: ALS Baseline correction (directly call als function)
    baseline, _ = whittaker.asls(smoothed)
    corrected = smoothed - baseline

    # Step 3: Peak detection (prominence-based)
    peaks, _ = find_peaks(corrected, prominence=min_prominence * np.max(corrected))

    # Step 4: Optional: Top-N peaks
    if top_n is not None and len(peaks) > top_n:
        top_indices = np.argsort(corrected[peaks])[-top_n:]
        peaks = peaks[np.sort(top_indices)]

    return mz_axis[peaks], corrected[peaks]"""

'''def detect_hdi_style_peaks(mz_axis, global_spectrum, 
                           window_length=9, polyorder=2, 
                           min_prominence=0.01, top_n=None,
                           save_path="detected_peaks.txt"):
    # Step 1: Savitzky-Golay smoothing
    smoothed = savgol_filter(global_spectrum, window_length=window_length, polyorder=polyorder)

    # Step 2: ALS Baseline correction
    baseline, _ = whittaker.asls(smoothed)
    corrected = smoothed - baseline

    # Step 3: Peak detection (prominence-based)
    peaks, _ = find_peaks(corrected, prominence=min_prominence * np.max(corrected))

    # Step 4: Optional: Top-N peaks
    if top_n is not None and len(peaks) > top_n:
        top_indices = np.argsort(corrected[peaks])[-top_n:]
        peaks = peaks[np.sort(top_indices)]

    # Extract m/z and intensity
    mz_peaks = mz_axis[peaks]
    intensities = corrected[peaks]

    # Sort by intensity (descending)
    sorted_indices = np.argsort(intensities)[::-1]
    mz_sorted = mz_peaks[sorted_indices]
    intensity_sorted = intensities[sorted_indices]

    # Save to text file
    np.savetxt(save_path, np.column_stack((mz_sorted, intensity_sorted)), 
               fmt="%.6f", header="m/z\tintensity", delimiter="\t")

    return mz_sorted, intensity_sorted'''

'''
def detect_hdi_style_peaks(mz_axis, global_spectrum, 
                           window_length=9, polyorder=2, 
                           min_prominence=0.01, top_n=None,
                           mz_tolerance=0.01,
                           save_path="detected_peaks.txt"):
    # Step 1: Savitzky-Golay smoothing
    smoothed = savgol_filter(global_spectrum, window_length=window_length, polyorder=polyorder)

    # Step 2: ALS Baseline correction
    baseline, _ = whittaker.asls(smoothed)
    corrected = smoothed - baseline

    # Step 3: Peak detection (prominence-based)
    peaks, _ = find_peaks(corrected, prominence=min_prominence * np.max(corrected))

    # Step 4: Optional: Top-N peaks
    if top_n is not None and len(peaks) > top_n:
        top_indices = np.argsort(corrected[peaks])[-top_n:]
        peaks = peaks[np.sort(top_indices)]

    # Extract m/z and intensity
    mz_peaks = mz_axis[peaks]
    raw_intensities = corrected[peaks]

    # Step 5: Sum intensities in ±mz_tolerance around each peak
    summed_intensities = []
    for mz in mz_peaks:
        mask = (mz_axis >= mz - mz_tolerance) & (mz_axis <= mz + mz_tolerance)
        summed_intensities.append(np.sum(corrected[mask]))
    summed_intensities = np.array(summed_intensities)

    # Step 6: Sort by summed intensity (descending)
    sorted_indices = np.argsort(summed_intensities)[::-1]
    mz_sorted = mz_peaks[sorted_indices]
    intensity_sorted = summed_intensities[sorted_indices]

    # Save to text file
    np.savetxt(save_path, np.column_stack((mz_sorted, intensity_sorted)), 
               fmt="%.6f", header="m/z\tintensity_sum_±0.01Da", delimiter="\t")

    return mz_sorted, intensity_sorted

def detect_prominence_peaks(mz_axis, global_spectrum, min_prominence=0.01, smoothing_sigma=None):
    if smoothing_sigma is not None and smoothing_sigma > 0:
        from scipy.ndimage import gaussian_filter1d
        spectrum = gaussian_filter1d(global_spectrum, sigma=smoothing_sigma)
    else:
        spectrum = global_spectrum

    peaks, _ = signal.find_peaks(spectrum, prominence=min_prominence * np.max(spectrum))
    return mz_axis[peaks], spectrum[peaks]


def detect_top_n_global_peaks(mz_axis, global_spectrum, top_n=1000, smoothing_sigma=1, min_da=0.05):
    smoothed = gaussian_filter1d(global_spectrum, sigma=smoothing_sigma)

    if smoothed.ndim != 1 or smoothed.size <= 1:
        raise ValueError("Global spectrum smoothing failed or is too small.")

    # Sort by intensity descending
    sorted_indices = np.argsort(smoothed)[::-1]
    mz_sorted = np.asarray(mz_axis)[sorted_indices]
    intensity_sorted = smoothed[sorted_indices]

    selected_mz = []
    selected_intensities = []

    for mz, intensity in zip(mz_sorted, intensity_sorted):
        if all(abs(mz - sel_mz) >= min_da for sel_mz in selected_mz):
            selected_mz.append(mz)
            selected_intensities.append(intensity)
            if len(selected_mz) >= top_n:
                break

    return np.array(selected_mz), np.array(selected_intensities)'''

def detect_hdi_style_peaks(mz_axis, global_spectrum, 
                           window_length=9, polyorder=2, 
                           min_prominence=0.01, top_n=None,
                           mz_tolerance=0.01,
                           save_path="detected_peaks.txt"):
    # Step 1: Savitzky-Golay smoothing
    smoothed = savgol_filter(global_spectrum, window_length=window_length, polyorder=polyorder)

    # Step 2: ALS Baseline correction
    baseline, _ = whittaker.asls(smoothed)
    corrected = smoothed - baseline

    # Step 3: Peak detection (prominence-based)
    peaks, _ = find_peaks(corrected, prominence=min_prominence * np.max(corrected))

    # Step 4: Optional: Top-N peaks
    if top_n is not None and len(peaks) > top_n:
        top_indices = np.argsort(corrected[peaks])[-top_n:]
        peaks = peaks[np.sort(top_indices)]

    # Extract m/z and intensities
    mz_peaks = mz_axis[peaks]
    raw_intensities = corrected[peaks]

    # Step 5: Sum intensities in ±mz_tolerance around each peak
    summed_intensities = []
    for mz in mz_peaks:
        mask = (mz_axis >= mz - mz_tolerance) & (mz_axis <= mz + mz_tolerance)
        summed_intensities.append(np.sum(corrected[mask]))
    summed_intensities = np.array(summed_intensities)

    # Step 6: Sort by summed intensity (descending)
    sorted_indices = np.argsort(summed_intensities)[::-1]
    mz_sorted = mz_peaks[sorted_indices]
    intensity_sorted = summed_intensities[sorted_indices]

    # Step 7: Remove overlapping peaks
    selected_mz = []
    selected_intensities = []
    for mz, intensity in zip(mz_sorted, intensity_sorted):
        if all(abs(mz - sel_mz) > mz_tolerance for sel_mz in selected_mz):
            selected_mz.append(mz)
            selected_intensities.append(intensity)

    # Convert to arrays
    selected_mz = np.array(selected_mz)
    selected_intensities = np.array(selected_intensities)

    # Step 8: Save to file
    np.savetxt(save_path, np.column_stack((selected_mz, selected_intensities)), 
               fmt="%.6f", header="m/z\tintensity_sum_±0.01Da", delimiter="\t")

    return selected_mz, selected_intensities

"""def extract_intensity_within_tolerance(adata, global_peaks, da_tolerance=0.015):
    mz_axis = adata.var_names.astype(float).values
    X = adata.X

    extracted = []
    new_var_names = []

    print(f"Total peaks passed to extraction: {len(global_peaks)}")

    for mz_target in tqdm(global_peaks, desc="Extracting peaks"):
        lower = mz_target - da_tolerance / 2
        upper = mz_target + da_tolerance / 2
        mask = (mz_axis >= lower) & (mz_axis <= upper)

        if np.any(mask):
            summed = X[:, mask].sum(axis=1)
            extracted.append(np.asarray(summed).flatten())
            new_var_names.append(f"{mz_target:.5f}")

    print(f"Total matched peaks after tolerance filtering: {len(extracted)}")

    new_X = np.vstack(extracted).T
    return ad.AnnData(X=new_X, obs=adata.obs.copy(), var=pd.DataFrame(index=new_var_names))"""

def extract_intensity_within_tolerance(adata, global_peaks, da_tolerance=0.015):
    mz_axis = adata.var_names.astype(float).values
    X = adata.X

    extracted = []
    new_var_names = []

    print(f"Total peaks passed to extraction: {len(global_peaks)}")

    used_indices = set()

    for mz_target in tqdm(global_peaks, desc="Extracting peaks"):
        lower = mz_target - da_tolerance / 2
        upper = mz_target + da_tolerance / 2
        mask = (mz_axis >= lower) & (mz_axis <= upper)
        indices = np.where(mask)[0]

        if len(indices) == 0:
            continue

        # Check if any of these indices have already been used
        if any(idx in used_indices for idx in indices):
            continue  # Skip this peak due to overlap

        # Mark indices as used
        used_indices.update(indices)

        summed = X[:, indices].sum(axis=1)
        extracted.append(np.asarray(summed).flatten())
        new_var_names.append(f"{mz_target:.5f}")

    print(f"Total matched (non-overlapping) peaks after filtering: {len(extracted)}")

    new_X = np.vstack(extracted).T
    return ad.AnnData(X=new_X, obs=adata.obs.copy(), var=pd.DataFrame(index=new_var_names))



In [2]:
# === Parameters ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_hdi_style_1000_0.04_100.h5ad"
method = "top_n"  # Options: "top_n" or "prominence"

# Common Parameters
smoothing_sigma = 1.5
da_tolerance = 0.04

# Method-specific
top_n = 1000
prominence = 0.001

intensity_threshold = 0
min_da=0.00

# === Load and Process ===
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

"""X = adata.X
X = X.toarray() if hasattr(X, "toarray") else X
row_sums = X.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # avoid division by zero
X_norm = X / row_sums

mz_axis, global_spec = create_global_spectrum(X_norm, adata.var_names.astype(float).values)"""


mz_axis, global_spec = create_global_spectrum_avg(adata.X, adata.var_names.astype(float).values)



"""
mz_axis, global_spec = create_global_spectrum(adata.X, adata.var_names.astype(float).values)"""
"""
import plotly.graph_objects as go

fig = go.Figure()

for mz, intensity in zip(mz_axis, global_spec):
    fig.add_trace(go.Scatter(
        x=[mz, mz],
        y=[0, intensity],
        mode='lines',
        line=dict(color='blue', width=1),
        showlegend=False
    ))

fig.update_layout(
    title='HDI-style Global Peak Spectrum',
    xaxis_title='m/z',
    yaxis_title='Intensity',
    template='plotly_white',
    height=500
)

fig.show()
"""
# === Peak Detection Switch ===
"""if method == "top_n":
    global_peaks, intensities = detect_top_n_global_peaks(
    mz_axis, global_spec, top_n=top_n, smoothing_sigma=smoothing_sigma, min_da=min_da
    )

elif method == "prominence":
    global_peaks = detect_prominence_peaks(
        mz_axis, global_spec, min_prominence=prominence, smoothing_sigma=smoothing_sigma
    )
    intensities = np.interp(global_peaks, mz_axis, gaussian_filter1d(global_spec, sigma=smoothing_sigma))
else:
    raise ValueError("Unknown method. Use 'top_n' or 'prominence'.")
"""
#peak_mz, peak_intensities = detect_prominence_peaks(mz_axis, global_spec, min_prominence=prominence, smoothing_sigma=smoothing_sigma)
peak_mz, peak_intensities = detect_hdi_style_peaks(mz_axis, global_spec, 
                                                    window_length=9, polyorder=2, 
                                                    min_prominence=0.01, top_n=None,
                                                    mz_tolerance=0.04, 
                                                    save_path="first_top_peaks_0100_040_sum.txt")



print(f'Number of peaks: {len(peak_mz)}')
top_n = 1000
sorted_indices = np.argsort(peak_intensities)[::-1][:top_n]
global_peaks = peak_mz[sorted_indices]
intensities = peak_intensities[sorted_indices]

import plotly.graph_objects as go

fig = go.Figure()

for mz, intensity in zip(global_peaks, intensities):
    fig.add_trace(go.Scatter(
        x=[mz, mz],
        y=[0, intensity],
        mode='lines',
        line=dict(color='blue', width=1),
        showlegend=False
    ))

fig.update_layout(
    title='HDI-style Global Peak Spectrum',
    xaxis_title='m/z',
    yaxis_title='Intensity',
    template='plotly_white',
    height=500
)

fig.show()
# === Filter by intensity threshold ===
mask = intensities >= intensity_threshold
global_peaks = global_peaks[mask]
intensities = intensities[mask]

print(f"Remaining peaks after intensity filtering (>{intensity_threshold}): {len(global_peaks)}")

# === Extract and Save ===
adata_hdi = extract_intensity_within_tolerance(adata, global_peaks, da_tolerance=da_tolerance)
adata_hdi.write(output_file)

print(f"Saved HDI-style AnnData to: {output_file}")
print(f"Number of top peaks detected: {len(global_peaks)}")
print("Some example peaks:", global_peaks[:10])


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad
Number of peaks: 138


Remaining peaks after intensity filtering (>0): 138
Total peaks passed to extraction: 138


Extracting peaks: 100%|██████████| 138/138 [01:25<00:00,  1.62it/s]

Total matched (non-overlapping) peaks after filtering: 138
Saved HDI-style AnnData to: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_hdi_style_1000_0.04_100.h5ad
Number of top peaks detected: 138
Some example peaks: [273.03970337 137.02508545 272.98922729 274.04348755 136.98420715
 331.02966309 369.35232544 273.22012329 257.04782104 195.09049988]


In [9]:
import plotly.graph_objects as go
import numpy as np

def plot_hdi_stem(adata_hdi, title="HDI-style Extracted Peaks"):
    # Get m/z values and max intensities across samples
    mz_axis = adata_hdi.var_names.astype(float).values
    if hasattr(adata_hdi.X, "toarray"):  # sparse matrix
        max_intensity = np.asarray(adata_hdi.X.max(axis=0)).ravel()
    else:
        max_intensity = np.max(adata_hdi.X, axis=0)

    # Sort by m/z for cleaner visualization
    sorted_idx = np.argsort(mz_axis)
    mz_sorted = mz_axis[sorted_idx]
    intensity_sorted = max_intensity[sorted_idx]

    fig = go.Figure()

    # Add vertical stem lines
    for x, y in zip(mz_sorted, intensity_sorted):
        fig.add_trace(go.Scatter(
            x=[x, x],
            y=[0, y],
            mode='lines',
            line=dict(color='crimson', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Add peak dots
    fig.add_trace(go.Scatter(
        x=mz_sorted,
        y=intensity_sorted,
        mode='markers',
        marker=dict(size=5, color='crimson'),
        hovertemplate='m/z: %{x:.6f}<br>Max Intensity: %{y:.2f}',
        name='Detected Peaks'
    ))

    fig.update_layout(
        title=title,
        xaxis_title='m/z',
        yaxis_title='Max Intensity (across samples)',
        template='plotly_white',
        hovermode='closest',
        dragmode='zoom'
    )

    fig.show()

In [10]:
plot_hdi_stem(adata_hdi)

In [10]:
plot_hdi_stem(adata_hdi)

In [24]:
plot_hdi_stem(adata_hdi)

In [29]:
import plotly.graph_objects as go

fig = go.Figure()

for mz, intensity in zip(global_peaks, intensities):
    fig.add_trace(go.Scatter(
        x=[mz, mz],
        y=[0, intensity],
        mode='lines',
        line=dict(color='blue', width=1),
        showlegend=False
    ))

fig.update_layout(
    title='HDI-style Global Peak Spectrum',
    xaxis_title='m/z',
    yaxis_title='Intensity',
    template='plotly_white',
    height=500
)

fig.show()

In [27]:
import plotly.graph_objects as go

fig = go.Figure()

for mz, intensity in zip(global_peaks, intensities):
    fig.add_trace(go.Scatter(
        x=[mz, mz],
        y=[0, intensity],
        mode='lines',
        line=dict(color='blue', width=1),
        showlegend=False
    ))

fig.update_layout(
    title='HDI-style Global Peak Spectrum',
    xaxis_title='m/z',
    yaxis_title='Intensity',
    template='plotly_white',
    height=500
)

fig.show()

In [ ]:
plot_hdi_stem(adata_hdi)

In [11]:
plot_hdi_stem(adata_hdi)


In [ ]:
plot_hdi_stem(adata_hdi)

In [5]:
plot_hdi_stem(adata_hdi)


In [12]:
plot_hdi_stem(adata_hdi)


In [8]:
plot_hdi_stem(adata_hdi)
